# CardioScore Validation 01 — Blinova/CiPA Summary Audit

This validates CiPA component semantics. It is not a five-endpoint CardioScore run.

In [ ]:

import sys, subprocess, json, hashlib, zipfile, tarfile
import pandas as pd
from pathlib import Path
PIN = "869150cd5fb5ccf155fb066258404bd4df163ade"
REPO = "Virelion-Biotech/Virelion-CardioScore"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"git+https://github.com/{REPO}.git@{PIN}"], check=True)
print("Installed pinned CardioScore:", PIN)


In [ ]:

from google.colab import files
up = files.upload()
path = Path(next(iter(up)))
df = pd.read_excel(path)
required = {"Drug_Name","Cell_type","risk","Platform","Type_of_EADs","conc","EAD","ddFPDc","site"}
missing = sorted(required - set(df.columns))
assert not missing, f"Missing required Blinova columns: {missing}"
raw_names = df["Drug_Name"].astype(str).str.strip()
canonical_names = (
    raw_names.str.lower()
    .str.replace(r"d[\\s,.-]*l[\\s,.-]*sotalol", "sotalol", regex=True)
    .str.replace(r"dl[\\s,.-]*sotalol", "sotalol", regex=True)
)
assert canonical_names.nunique() == 28, f"Expected 28 canonical compounds after documented sotalol-name normalization; found {canonical_names.nunique()}. Unique raw names: {sorted(raw_names.unique())}"
assert df["site"].nunique() == 10, f"Expected 10 sites, found {df['site'].nunique()}"
assert pd.to_numeric(df["conc"], errors="coerce").notna().all()
assert pd.to_numeric(df["ddFPDc"], errors="coerce").notna().all()
print("rows:", len(df), "drugs:", df["Drug_Name"].nunique(), "sites:", df["site"].nunique())


In [ ]:

risk_map = {"l":"low","m":"intermediate","h":"high"}
df = df.copy()
df["compound"] = df["Drug_Name"].astype(str).str.strip().str.lower().str.replace("d,l sotalol","sotalol",regex=False).str.replace("dl sotalol","sotalol",regex=False)
df["reference_risk"] = df["risk"].astype(str).str.strip().str.lower().map(risk_map)
assert df["reference_risk"].notna().all(), "Unexpected risk labels; stop rather than inventing a mapping."
platforms = set(df["Platform"].astype(str).str.strip())
allowed = {"AXN","CLY","ECR","AMD","MCS"}
unexpected = sorted(platforms - allowed)
assert not unexpected, f"Unexpected platform code(s): {unexpected}. Preserve and investigate; do not silently relabel."
event = df["Type_of_EADs"].astype(str).str.strip().str.upper()
broad = pd.to_numeric(df["EAD"], errors="coerce")
assert broad.notna().all()
df["is_abcd_arrhythmia"] = event.isin({"A","B","C","D"})
df["is_Q_quiescence"] = event.eq("Q")
assert (df["is_abcd_arrhythmia"] | df["is_Q_quiescence"]).le(broad.eq(1)).all()
print("A-D rows:", int(df["is_abcd_arrhythmia"].sum()), "Q rows:", int(df["is_Q_quiescence"].sum()))


In [ ]:

for drug in ["verapamil","terfenadine"]:
    m = df["compound"].eq(drug)
    if m.any():
        abcd = int(df.loc[m, "is_abcd_arrhythmia"].sum())
        q = int(df.loc[m, "is_Q_quiescence"].sum())
        print(drug, "A-D:", abcd, "Q:", q, "broad EAD:", int(df.loc[m,"EAD"].eq(1).sum()))
        assert abcd == 0, f"{drug} has an unexpected A-D event in this workbook; inspect source semantics."

reference = df[["compound","reference_risk"]].drop_duplicates()
assert reference.groupby("compound").size().max() == 1
reference.to_csv("/content/cardioscore_validation/derived/blinova_reference.csv", index=False)
semantic = df[["compound","Drug_Name","risk","Platform","site","Cell_type","conc","ddFPDc","EAD","Type_of_EADs","is_abcd_arrhythmia","is_Q_quiescence"]]
semantic.to_csv("/content/cardioscore_validation/derived/blinova_semantic_summary.csv", index=False)
print(reference.to_string(index=False))


### Interpretation
`Q` is handled separately from A–D arrhythmia-like events. `ddFPDc` is retained as its published quantity and is not substituted for CardioScore `fpd_change_pct`.